In [1]:
import torch
print(torch.__version__) # make sure it's 2.7.0 to import wav2vec
print(torch.__file__)

2.7.0+cu126
d:\anaconda\envs\deepfake\Lib\site-packages\torch\__init__.py


In [2]:
from transformers import Wav2Vec2Model, Wav2Vec2Processor
import librosa
import os
from tqdm import tqdm
import numpy as np
import torch.nn.functional as F

d:\anaconda\envs\deepfake\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# load the pretrained model
model_name = 'facebook/wav2vec2-base'
pre_processor = Wav2Vec2Processor.from_pretrained(model_name)
model = Wav2Vec2Model.from_pretrained(model_name)

d:\anaconda\envs\deepfake\Lib\site-packages\transformers\configuration_utils.py:312: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

Wav2Vec2Model(
  (feature_extractor): Wav2Vec2FeatureEncoder(
    (conv_layers): ModuleList(
      (0): Wav2Vec2GroupNormConvLayer(
        (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
        (activation): GELUActivation()
        (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
      )
      (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
      (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
    )
  )
  (feature_projection): Wav2Vec2FeatureProjection(
    (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (projection): Linear(in_features=512, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): Wav2Vec2Encoder(
    (pos_conv_embed): Wav2Vec2PositionalConvEmbedding(
  

In [10]:
# Label mapping
label_map = {'bonafide': 1, 'spoof': 0}
audio_labels = {}

# === Paths ===
protocol_path = "D:\\Stuttgart\\(important) Third Semester\\Team lab\\LA\\ASVspoof2019_LA_cm_protocols\\ASVspoof2019.LA.cm.eval.trl.txt"
audio_base_path = "D:\\Stuttgart\\(important) Third Semester\\Team lab\\LA\\ASVspoof2019_LA_eval\\flac"
output_tensor_dir = "D:\\Stuttgart\\(important) Third Semester\\Team lab\\wav2vec_eval"
os.makedirs(output_tensor_dir, exist_ok=True)

# === Load protocol labels ===
with open(protocol_path, 'r') as file:
    for line in file:
        parts = line.strip().split()
        file_name = parts[1]
        label = parts[-1]
        audio_labels[file_name] = label_map[label]

# === Utility functions ===
def limit_audio_length(y, sr, target_length=4):
    target_samples = target_length * sr
    if len(y) > target_samples:
        return y[:target_samples]
    else:
        repeats = target_samples // len(y) + 1
        return np.tile(y, repeats)[:target_samples]

def get_wav2vec_tensor(audio_path):
    y, sr = librosa.load(audio_path, sr=16000)
    y = limit_audio_length(y, sr)
    inputs = pre_processor(y, sampling_rate=sr, return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model(**inputs).last_hidden_state  # [B, T, D]

        # Normalize each time step (optional)
        output = F.layer_norm(output, output.shape[1:])  # normalize over dim=768

        # Reshape to [B, 1, 768, T]
        wav2vec_tensor = output.permute(0, 2, 1).unsqueeze(1)  # [B, 1, 768, T]

    return wav2vec_tensor.cpu()

# === Process audio files ===
audio_files = [f for f in os.listdir(audio_base_path) if f.endswith(".flac")]

for f in tqdm(audio_files, desc="Extracting and labeling"):
    file_path = os.path.join(audio_base_path, f)
    file_id = os.path.splitext(f)[0]

    if file_id not in audio_labels:
        print(f"Warning: No label found for {file_id}, skipping.")
        continue

    try:
        wav2vec_tensor = get_wav2vec_tensor(file_path)
        label = audio_labels[file_id]
        tensor_output_path = os.path.join(output_tensor_dir, f"{file_id}.pt")
        torch.save((wav2vec_tensor, label), tensor_output_path)
    except RuntimeError as e:
        print(f"Failed on {file_id}: {e}")
        torch.cuda.empty_cache()

print("✅ All audio processed and saved with labels.")
    
    






Extracting and labeling: 100%|██████████| 71933/71933 [37:18<00:00, 32.14it/s]    

✅ All audio processed and saved with labels.


In [9]:
# === Process audio files ===
audio_files = [f for f in os.listdir(audio_base_path) if f.endswith(".flac")]

for f in tqdm(audio_files, desc="Extracting and labeling"):
    file_path = os.path.join(audio_base_path, f)
    file_id = os.path.splitext(f)[0]

    if file_id not in audio_labels:
        print(f"Warning: No label found for {file_id}, skipping.")
        continue

    wav2vec_tensor = get_wav2vec_tensor(file_path).cpu()
    label = audio_labels[file_id]

    tensor_output_path = os.path.join(output_tensor_dir, f"{file_id}.pt")
    torch.save((wav2vec_tensor, label), tensor_output_path)

print("✅ All audio processed and saved with labels.")

Extracting and labeling: 100%|██████████| 24986/24986 [10:16<00:00, 40.52it/s]

✅ All audio processed and saved with labels.


(199, 768)